In [3]:
import pandas as pd
import numpy as np

In [4]:
print("Loading optimized data...")

model_data = pd.read_parquet("../data/processed/preprocessed_data_optimized.parquet")



Loading optimized data...


In [6]:
# "Days Since Last Sale" feature to be added

model_data["sale_date"] = model_data['date'].where(model_data['sales'] > 0)

# forward filling these dates per item (propagating the last sale date forward)
model_data['last_sale_date'] = model_data.groupby('item_id')['sale_date'].ffill()

model_data['shifted_last_sale_date'] = model_data.groupby('item_id')['last_sale_date'].shift(1)

model_data['days_since_last_sale'] = (model_data['date'] - model_data['shifted_last_sale_date']).dt.days

model_data['days_since_last_sale'] = model_data['days_since_last_sale'].fillna(999).astype('int16')


model_data.drop(columns=['sale_date', 'last_sale_date', 'shifted_last_sale_date'], inplace=True)

In [7]:
print("Engineering 2: Target Mean Encodings...")

model_data['item_mean_sales'] = (
    model_data.groupby('item_id')['sales']
    .transform(lambda x: x.shift(1).expanding().mean())
    .astype('float32')
)

model_data['item_wday_mean'] = (
    model_data.groupby(['item_id', 'wday'])['sales']
    .transform(lambda x: x.shift(1).expanding().mean())
    .astype('float32')
)

model_data['item_mean_sales'] = model_data['item_mean_sales'].fillna(0)
model_data['item_wday_mean'] = model_data['item_wday_mean'].fillna(0)

Engineering 2: Target Mean Encodings...


In [8]:
print("Saving enriched dataset...")

save_path = "../data/processed/preprocessed_data_enriched.parquet"
model_data.to_parquet(save_path, index=False)
print(f"Success! Enriched data saved to {save_path}")

Saving enriched dataset...
Success! Enriched data saved to ../data/processed/preprocessed_data_enriched.parquet
